# Text-to-Video Generation with stable-diffusion.cpp

This notebook demonstrates how to generate videos from text prompts using stable-diffusion.cpp.

## What This Notebook Does

1. Loads a pre-trained video generation model
2. Takes your text prompt
3. Generates a video frame-by-frame
4. Saves the output as a video file

## Requirements

- Python 3.8+
- stable-diffusion.cpp
- ffmpeg (for video encoding)

## Hardware Requirements

- **Minimum**: 16 GB RAM (smaller models only, low quality)
- **Recommended**: 24 GB+ RAM or 10 GB+ VRAM
- **Optimal**: 32 GB+ RAM or 16 GB+ VRAM

## Model Selection Guide

| RAM/VRAM | Model | Quality | Speed |
|----------|-------|---------|-------|
| 16 GB+ | odyssey-systems/Wan2.1-T2V-1.3B-bf16 | Low | Fast |
| 20 GB+ | Wan-AI/Wan2.1-T2V-1.3B | Medium | Medium |
| 24 GB+ | stabilityai/stable-video-diffusion-img2vid-xt | High | Slow |

## How It Works (Simple Explanation)

1. **Text Encoder**: Converts your text prompt into numbers (embeddings)
2. **Noise Scheduler**: Starts with random noise
3. **U-Net Model**: Repeatedly cleans the noise, guided by your text
4. **Frame Generation**: Creates each video frame step-by-step
5. **Video Assembly**: Combines frames into a video file

## Getting Help

If you see errors:
1. Check you you have enough RAM/VRAM
2. Try a smaller model
3. Reduce the video length or resolution
4. Check the documentation in `docs/` folder

In [ ]:
# Step 1: Import required libraries
import sys
import os

# Add scripts directory to path
sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', 'scripts'))

from detect_hardware import detect_hardware
from recommend_models import recommend_models

print("Libraries imported successfully")

# Run hardware detection
hw = detect_hardware()
print(f"Platform: {hw['system']['platform']} {hw['system']['platform_release']}")
print(f"RAM: {hw['ram']['total_gb']} GB")
print(f"Storage: {hw['storage']['free_gb']} GB free")

In [ ]:
# Step 2: Load the model
# This will download the model on first run (~3-5GB)

def load_model(model_name="odyssey-systems/Wan2.1-T2V-1.3B-bf16"):
    """Load a video generation model""
    
    print(f"Loading model: {model_name}")
    print(f"This will download the model on first run (~3-5GB)")
    print("Please wait...")
    
    try:
        # Import stable-diffusion.cpp
        import sd2cpp
        
        # Load the pipeline
        pipe = sd2cpp.VideoGenerationPipeline(model_name=model_name)
        print(f"Model loaded successfully!")
        return pipe
    except Exception as e:
        print(f"Error loading model: {e}")
        print("\nCommon issues:")
        print("1. Not enough GPU memory - try smaller model or CPU")
        print("2. Internet connection needed for first download")
        print("3. Install stable-diffusion.cpp: bash scripts/install_local_video.sh")
        raise

# Load model (change model_name for different quality/size tradeoffs)
model_name = "odyssey-systems/Wan2.1-T2V-1.3B-bf16"  # Change this for different models
pipe = load_model(model_name)

In [ ]:
# Step 3: Generate video from text

def generate_video(pipe, prompt, num_frames=16, height=256, width=256):
    """Generate a video from a text prompt""
    
    print(f"\nGenerating video...")
    print(f"Prompt: {prompt}")
    print(f"Frames: {num_frames}, Size: {height}x{width}")
    print("This may take a few minutes...")
    
    try:
        # Generate the video frames
        frames = pipe.generate(
            prompt,
            num_frames=num_frames,
            height=height,
            width=width,
            num_inference_steps=50,  # More steps = better quality, slower
        )
        
        print(f"\nGenerated {len(frames)} frames!")
        return frames
        
    except Exception as e:
        print(f"\nError during generation: {e}")
        print("\nTry:")
        print("1. Reduce num_frames (try 8 instead 16)")
        print("2. Reduce height/width (try 128x128)")
        print("3. Use a smaller model")
        raise

# Your text prompt goes here
prompt = "A beautiful sunset in a coastal city with colorful scenery"

# Generate video (adjust parameters for your hardware)
frames = generate_video(
    pipe,
    prompt,
    num_frames=16,    # Number of frames (8=short, 16=medium, 32=long)
    height=256,       # Video height (128=small, 256=medium, 512=large)
    width=256         # Video width (same as height for square)
)

In [ ]:
# Step 4: Save the generated video

import cv2
import os

def save_video(frames, prompt, output_dir="output"):
    """Save generated frames as a video file""
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Create filename from prompt
    safe_prompt = ''.join(c if c.isalnum() or c in ' -_' else '_' for c in prompt[:50])
    output_path = os.path.join(output_dir, f"video_{safe_prompt}.mp4")
    
    if not frames:
        print("No frames to save!")
        return None
    
    # Get frame dimensions
    first_frame = frames[0]
    if hasattr(first_frame, 'size'):
        height, width = first_frame.size[1], first_frame.size[0]
    else:
        height, width = first_frame.shape[:2]
    
    # Create video writer
    fourcc = cv2.VideoWriter_fourcc('mp4v')
    video = cv2.VideoWriter(output_path, fourcc, 4.0, (width, height))
    
    # Write each frame
    for i, frame in enumerate(frames):
        # Convert frame to proper format
        if hasattr(frame, 'convert'):
            frame = frame.convert('RGB')
        
        # Convert to numpy array if needed
        if hasattr(frame, 'numpy'):
            frame_np = frame.numpy()
        else:
            frame_np = np.array(frame)
        
        # Convert RGB to BGR for OpenCV
        frame_bgr = cv2.cvtColor(frame_np, cv2.COLOR_RGB2BGR)
        video.write(frame_bgr)
    
    video.release()
    print(f"\nVideo saved: {output_path}")
    return output_path

# Save the generated video
video_path = save_video(frames, prompt)
print(f"\nYour video is ready at: {video_path}")

## Next Steps

1. **Watch your video**: Open the saved file in your video player
2. **Try different prompts**: Change the `prompt` variable and run again
3. **Adjust quality**: Change `num_frames`, `height`, `width` for different results
4. **Try image-to-video**: Use the `image-to-video.ipynb` notebook

## Troubleshooting

### Out Memory Error
If you see "Out Memory" or "CUDA out memory":
- Reduce `num_frames` to 8
- Reduce `height`/`width` to 128
- Close other applications to free RAM

### Slow Generation
If generation is too slow:
- Use fewer frames (8 instead 16)
- Use lower resolution (128x128)
- Enable GPU if available

### Missing Packages
If you see import errors:
```bash
pip install opencv-python
```